# Smart Farming — Notebook 2: Vision + IndicTrans2 deployment

Runs **vision on port 8001** and **IndicTrans2 translation on port 8002** in one Colab runtime. The vision service uses the device logic in your repository; translation selects CUDA when `torch.cuda.is_available()` and otherwise uses CPU.

After creating new Cloudflare Quick Tunnels, this notebook updates the stable Worker gateway's KV values `VISION_URL` and `TRANSLATION_URL`.

**Prerequisites:** Run Notebook 1 once, configure the Cloudflare Worker/KV binding, and add Colab Secrets `HF_TOKEN`, `CF_API_TOKEN`, `CF_ACCOUNT_ID`, `CF_KV_NAMESPACE_ID`, `CF_WORKER_BASE_URL`, and `GATEWAY_API_KEY`.


## 1. Mount Drive and check files

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, time, signal, re, json, requests
DRIVE_ROOT = "/content/drive/MyDrive/SmartFarming"
assert os.path.isfile(os.path.join(DRIVE_ROOT, "main.py")), (
    f"main.py not found in {DRIVE_ROOT}. Run Notebook 1 and verify repository layout."
)
print("Service directory:", DRIVE_ROOT)


## 2. Install vision + translation dependencies

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN") or ""
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs("/content/hf_cache", exist_ok=True)

req = os.path.join(DRIVE_ROOT, "requirements.txt")
if os.path.isfile(req):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)
else:
    print("WARNING: requirements.txt not found in Drive; installing basic API dependencies.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn[standard]", "requests"], check=True)

# IndicTrans2 toolkit/model remote code compatibility.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.51,<5", "sentencepiece", "sacremoses"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/VarunGumma/IndicTransToolkit"], check=True)
print("Dependencies installed.")


## 3. Write IndicTrans2 API server into Drive

In [ ]:
%%writefile /content/drive/MyDrive/IndicTransServer/translate_server.py
import os, json, threading
os.environ.setdefault("HF_HOME", "/content/hf_cache")

import torch
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

LANGS = {"hi": "hin_Deva", "gu": "guj_Gujr"}
MODEL_NAME = "ai4bharat/indictrans2-en-indic-dist-200M"
CACHE_FILE = "/content/drive/MyDrive/indictrans_cache.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_BEAMS = 4 if DEVICE == "cuda" else 2
CHUNK = 16 if DEVICE == "cuda" else 8
if DEVICE == "cpu":
    torch.set_num_threads(os.cpu_count() or 2)
print(f"[Init] Device: {DEVICE}")

print("[Init] Loading IndicTrans2 tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=True).to(DEVICE).eval()
processor = IndicProcessor(inference=True)
lock = threading.Lock()

cache = {}
if os.path.exists(CACHE_FILE):
    try:
        with open(CACHE_FILE, encoding="utf-8") as f:
            cache = json.load(f)
        print(f"[Cache] Loaded {len(cache)} cached translations from Drive.")
    except Exception as e:
        print(f"[Cache] Could not load cache: {e}")

def save_cache():
    try:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False)
    except Exception:
        pass

def run(texts, tgt):
    batch = processor.preprocess_batch(texts, src_lang="eng_Latn", tgt_lang=tgt)
    inputs = tokenizer(batch, padding=True, truncation=True, max_length=256,
                       return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        out = model.generate(**inputs, num_beams=NUM_BEAMS, max_length=256, use_cache=False)
    return processor.postprocess_batch(
        tokenizer.batch_decode(out, skip_special_tokens=True), lang=tgt)

run(["Field health looks good."], "hin_Deva")
print("[Init] IndicTrans2 warmed up and ready.")

app = FastAPI(title="IndicTrans2 Translation Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

class TranslationRequest(BaseModel):
    texts: list[str]
    target: str

@app.get("/health")
def health():
    return {"status": "healthy", "service": "indictrans2", "device": DEVICE}

@app.post("/translate")
def translate(req: TranslationRequest):
    lang = req.target.lower()
    tgt = LANGS.get(lang)
    if not tgt:
        return {"translations": req.texts, "error": f"Unsupported target {req.target}"}

    def key(t):
        return f"{lang}:{t}"

    with lock:
        todo = [t for t in dict.fromkeys(req.texts) if key(t) not in cache]
        for i in range(0, len(todo), CHUNK):
            chunk = todo[i:i + CHUNK]
            for s, d in zip(chunk, run(chunk, tgt)):
                cache[key(s)] = d
        if todo:
            save_cache()
        return {"translations": [cache.get(key(t), t) for t in req.texts]}


## 4. Install Cloudflare Tunnel

In [ ]:
cloudflared_path = "/content/cloudflared"
if not os.path.isfile(cloudflared_path):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", cloudflared_path], check=True)
os.chmod(cloudflared_path, 0o755)
subprocess.run([cloudflared_path, "--version"], check=True)


## 5. Start vision and translation services (automatic CPU/GPU selection)

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN") or ""
env = {**os.environ, "HF_TOKEN": HF_TOKEN, "HF_HOME": "/content/hf_cache"}
os.makedirs("/content/hf_cache", exist_ok=True)

def stop_matching(pattern):
    out = subprocess.run(["bash", "-lc", f"pgrep -f {pattern!r} || true"],
                         capture_output=True, text=True).stdout.split()
    for raw_pid in out:
        try:
            pid = int(raw_pid)
            if pid != os.getpid():
                os.kill(pid, signal.SIGTERM)
        except Exception:
            pass

def start_service(name, cmd, cwd, log_path):
    log = open(log_path, "w")
    proc = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=log, stderr=subprocess.STDOUT)
    return proc, log

def wait_ready(name, url, proc, log_path, timeout=900):
    start_time = time.time()
    while time.time() - start_time < timeout:
        if proc.poll() is not None:
            print(f"{name} exited with code {proc.returncode}. Log tail:")
            print(open(log_path, errors="replace").read()[-5000:])
            return False
        try:
            response = requests.get(url, timeout=5)
            if response.status_code < 500:
                print(f"{name} ready in {time.time()-start_time:.1f}s (HTTP {response.status_code})")
                return True
        except requests.RequestException:
            pass
        time.sleep(3)
    print(f"{name} did not become ready in {timeout}s. Log tail:")
    print(open(log_path, errors="replace").read()[-5000:])
    return False

# Start vision API (device selection is handled by the repository's model loader).
vision_log = "/content/vision.log"
vision_proc, _vision_log_handle = start_service(
    "vision", ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8001"],
    DRIVE_ROOT, vision_log
)
vision_ok = wait_ready("vision", "http://127.0.0.1:8001/docs", vision_proc, vision_log)

# Start translation API; translate_server.py selects cuda if available, otherwise cpu.
translate_dir = "/content/drive/MyDrive/IndicTransServer"
os.makedirs(translate_dir, exist_ok=True)
translate_log = "/content/translate.log"
translate_proc, _translate_log_handle = start_service(
    "translation", ["uvicorn", "translate_server:app", "--host", "0.0.0.0", "--port", "8002"],
    translate_dir, translate_log
)
translate_ok = wait_ready("translation", "http://127.0.0.1:8002/health", translate_proc, translate_log)
assert vision_ok and translate_ok, "At least one service failed to start. Inspect /content/vision.log and /content/translate.log."


## 6. Create Quick Tunnels and update Cloudflare KV

In [ ]:
def open_tunnel(port, timeout=90):
    log_path = f"/content/tunnel_{port}.log"
    log_handle = open(log_path, "w")
    proc = subprocess.Popen(
        ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
        stdout=log_handle, stderr=subprocess.STDOUT
    )
    end = time.time() + timeout
    while time.time() < end:
        try:
            content = open(log_path, errors="replace").read()
        except Exception:
            content = ""
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            return proc, match.group(0), log_path
        if proc.poll() is not None:
            break
        time.sleep(1)
    raise RuntimeError(f"Could not obtain tunnel URL for port {port}. Log: {log_path}")

vision_tunnel_proc, VISION_URL, vision_tunnel_log = open_tunnel(8001)
translate_tunnel_proc, TRANSLATION_URL, translate_tunnel_log = open_tunnel(8002)

print("VISION_URL      =", VISION_URL)
print("TRANSLATION_URL =", TRANSLATION_URL)

# Save a convenience copy in Drive.
with open("/content/drive/MyDrive/service_urls_vision_translation.json", "w", encoding="utf-8") as f:
    json.dump({"vision": VISION_URL, "translation": TRANSLATION_URL}, f, indent=2)

import requests
from google.colab import userdata

def colab_secret(name, required=True):
    value = userdata.get(name)
    if required and not value:
        raise RuntimeError(
            f"Missing Colab Secret {name}. Open the key icon in Colab Secrets, add it, "
            "and enable notebook access."
        )
    return value

CF_API_TOKEN = colab_secret("CF_API_TOKEN")
CF_ACCOUNT_ID = colab_secret("CF_ACCOUNT_ID")
CF_KV_NAMESPACE_ID = colab_secret("CF_KV_NAMESPACE_ID")

def update_cloudflare_kv(key, value):
    """Create/overwrite one Worker KV value using the Cloudflare API."""
    endpoint = (
        f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}"
        f"/storage/kv/namespaces/{CF_KV_NAMESPACE_ID}/values/{key}"
    )
    response = requests.put(
        endpoint,
        headers={"Authorization": f"Bearer {CF_API_TOKEN}",
                 "Content-Type": "text/plain"},
        data=value.encode("utf-8"),
        timeout=30
    )
    try:
        payload = response.json()
    except Exception:
        payload = {}
    if not response.ok or payload.get("success") is False:
        raise RuntimeError(
            f"Cloudflare KV update failed for {key}: HTTP {response.status_code}; "
            f"{response.text[:500]}"
        )
    print(f"[Cloudflare KV] Updated {key} -> {value}")

update_cloudflare_kv("VISION_URL", VISION_URL)
update_cloudflare_kv("TRANSLATION_URL", TRANSLATION_URL)
print("\nStable gateway base URL:", colab_secret("CF_WORKER_BASE_URL"))
print("Routes: /vision/docs and /translation/health (use Authorization: Bearer GATEWAY_API_KEY)")


## 7. Test the local services and stable gateway

In [ ]:
from google.colab import userdata
GATEWAY_BASE = (userdata.get("CF_WORKER_BASE_URL") or "").rstrip("/")
GATEWAY_KEY = userdata.get("GATEWAY_API_KEY") or ""
assert GATEWAY_BASE and GATEWAY_KEY, "Add CF_WORKER_BASE_URL and GATEWAY_API_KEY to Colab Secrets."
auth = {"Authorization": f"Bearer {GATEWAY_KEY}"}

for label, url in [
    ("local vision", "http://127.0.0.1:8001/docs"),
    ("local translation", "http://127.0.0.1:8002/health"),
    ("gateway vision", f"{GATEWAY_BASE}/vision/docs"),
    ("gateway translation", f"{GATEWAY_BASE}/translation/health")
]:
    try:
        r = requests.get(url, headers=auth if url.startswith(GATEWAY_BASE) else {}, timeout=20)
        print(label, "HTTP", r.status_code, r.text[:250])
    except Exception as exc:
        print(label, "FAILED:", exc)


## 8. Diagnostics

In [ ]:
for log_path in ["/content/vision.log", "/content/translate.log",
                  "/content/tunnel_8001.log", "/content/tunnel_8002.log"]:
    print("\n====", log_path, "====")
    if os.path.isfile(log_path):
        print(open(log_path, errors="replace").read()[-4000:])
    else:
        print("Not found")
print("\nGPU status:")
subprocess.run(["bash", "-lc", "nvidia-smi || echo 'No GPU assigned; services should use CPU where supported.'"])
